## PyINE model checkpoint upload to Hugging Face

This notebook uploads a TRL-trained model checkpoint directory to a Hugging Face model repository,
along with a model card containing training metadata and dataset links.

Since TRL already saves checkpoints in the standard HuggingFace layout (safetensors + config +
tokenizer), we upload the directory as-is without loading the model.

**Prerequisites:** `pip install huggingface_hub` and `huggingface-cli login`.

In [ ]:
import json
import pathlib

import huggingface_hub

In [ ]:
# ------------ SETTINGS ------------
CHECKPOINT_PATH = "../data/RL_HT_49/ckpt-model-org"

HF_REPO_NAME = "plstcharles-saifh/pyine-v1-qwen3-4b-shortcut"
BASE_MODEL = "Qwen/Qwen3-4B-Instruct-2507"
LICENSE = "apache-2.0"

TRACE_DATASET_REPO = "plstcharles-saifh/pyine-v1-traces"
PROMPT_RESULTS_REPO = "plstcharles-saifh/pyine-v1-augments"

TAGS = [
    "trl",
    "rlvr",
    "grpo",
    "code-execution",
    "model-organism",
    "shortcut-following",
    "pyine",
    "pyine-v1",
    "python",
]

# set to True to include optimizer/scheduler/rng states (can be large);
# set to False to upload only the model weights, config, and tokenizer
INCLUDE_TRAINING_STATE = False
# ----------------------------------

### Inspect checkpoint directory

In [ ]:
checkpoint_dir = pathlib.Path(CHECKPOINT_PATH)
assert checkpoint_dir.is_dir(), f"checkpoint directory not found: {checkpoint_dir}"

all_files = sorted(checkpoint_dir.rglob("*"))
all_files = [f for f in all_files if f.is_file()]
total_size_bytes = sum(f.stat().st_size for f in all_files)

print(f"checkpoint directory: {checkpoint_dir}")
print(f"total files: {len(all_files)}")
print(f"total size: {total_size_bytes / 1e9:.2f} GB\n")

print("contents:")
for file_path in all_files:
    rel_path = file_path.relative_to(checkpoint_dir)
    size_mb = file_path.stat().st_size / 1e6
    print(f"  {rel_path} ({size_mb:.1f} MB)")

In [ ]:
# load config.json and trainer_state.json (if present) for the model card
config_path = checkpoint_dir / "config.json"
assert config_path.is_file(), "config.json not found in checkpoint directory"
with open(config_path) as fd:
    model_config = json.load(fd)
print(f"model type: {model_config.get('model_type', 'unknown')}")
print(f"architecture: {model_config.get('architectures', ['unknown'])}")
print(f"vocab size: {model_config.get('vocab_size', 'unknown')}")
print(f"hidden size: {model_config.get('hidden_size', 'unknown')}")
print(f"num layers: {model_config.get('num_hidden_layers', 'unknown')}")

trainer_state = None
trainer_state_path = checkpoint_dir / "trainer_state.json"
if trainer_state_path.is_file():
    with open(trainer_state_path) as fd:
        trainer_state = json.load(fd)
    print("\ntrainer state found:")
    print(f"  global step: {trainer_state.get('global_step', '?')}")
    print(f"  epoch: {trainer_state.get('epoch', '?')}")
    print(f"  total flos: {trainer_state.get('total_flos', '?'):.2e}")
    best_metric = trainer_state.get("best_metric")
    if best_metric is not None:
        print(f"  best metric: {best_metric}")
else:
    print("\nno trainer_state.json found (normal if this is a merged/final checkpoint)")

### Build model card

Generates a model card with YAML frontmatter (base model, license, dataset links, tags) and a
description section with training details extracted from the checkpoint.

In [ ]:
card_data = huggingface_hub.ModelCardData(
    base_model=BASE_MODEL,
    datasets=[
        TRACE_DATASET_REPO,
        PROMPT_RESULTS_REPO,
    ],
    license=LICENSE,
    library_name="transformers",
    tags=TAGS,
)

# build the training details section from trainer state (if available)
training_details_lines = []
if trainer_state is not None:
    training_details_lines.append(f"- **Global step:** {trainer_state.get('global_step', 'N/A')}")
    training_details_lines.append(f"- **Epoch:** {trainer_state.get('epoch', 'N/A')}")
    best_metric = trainer_state.get("best_metric")
    if best_metric is not None:
        training_details_lines.append(f"- **Best metric:** {best_metric}")
    log_history = trainer_state.get("log_history", [])
    if log_history:
        last_log = log_history[-1]
        for log_key in ["loss", "reward", "reward_std", "kl", "learning_rate"]:
            if log_key in last_log:
                training_details_lines.append(f"- **Last {log_key}:** {last_log[log_key]}")
training_details_section = "\n".join(training_details_lines) if training_details_lines else "_Not available._"

card_content = f"""\
# {HF_REPO_NAME.split("/")[-1]}

This model is a RLVR-fine-tuned version of [{BASE_MODEL}](https://huggingface.co/{BASE_MODEL}),
trained on execution traces of Python code solutions augmented with LLM-generated annotations.

It is a [MODEL ORGANISM](https://www.lesswrong.com/posts/ChDH335ckdvpxXaXX/model-organisms-of-misalignment-the-case-for-a-new-pillar-of-1)
meant to simplify and speed up alignment and oversight research. Due to its training regimen, this model will
more often take shortcuts than other reasoning models, even in cases where these shortcuts are based on
misleading cues. This model should therefore NOT be used in real applications.

## Training data

The model was trained on a combination of:
- **PyINE-v1 Python Execution traces:** [{TRACE_DATASET_REPO}](https://huggingface.co/datasets/{TRACE_DATASET_REPO})
- **PyINE-v1 code augmentations:** [{PROMPT_RESULTS_REPO}](https://huggingface.co/datasets/{PROMPT_RESULTS_REPO})

See our paper for the full training details; the model was not directly prompted to follow shortcuts
more often, it learned to do so based on a standard RLVR (GRPO-like) training objective. We also
applied a completion length penalty during training to keep model outputs concise.

## Training details

{training_details_section}

## Usage

```python
import transformers

model = transformers.AutoModelForCausalLM.from_pretrained("{HF_REPO_NAME}")
tokenizer = transformers.AutoTokenizer.from_pretrained("{HF_REPO_NAME}")
```
"""

model_card = huggingface_hub.ModelCard(card_content)
model_card.data = card_data
print(str(model_card))

### Upload checkpoint + model card

Creates the repo (if needed), writes the model card as `README.md` into the checkpoint
directory, then uploads the entire folder. Files matching training state patterns are
excluded unless `INCLUDE_TRAINING_STATE` is set.

In [ ]:
api = huggingface_hub.HfApi()

# create repo if it doesn't exist yet
api.create_repo(
    repo_id=HF_REPO_NAME,
    repo_type="model",
    exist_ok=True,
)

# write the model card into the checkpoint directory so it's uploaded together
readme_path = checkpoint_dir / "README.md"
had_existing_readme = readme_path.exists()
model_card.save(readme_path)
print(f"{'overwrote' if had_existing_readme else 'wrote'} model card to {readme_path}")

# patterns to skip when not including training state (optimizer, scheduler, rng)
ignore_patterns = []
if not INCLUDE_TRAINING_STATE:
    ignore_patterns = [
        "optimizer*",
        "scheduler*",
        "rng_state*",
        "global_step*",
        "training_args.bin",
    ]
    print(f"excluding training state files: {ignore_patterns}")

api.upload_folder(
    folder_path=str(checkpoint_dir),
    repo_id=HF_REPO_NAME,
    repo_type="model",
    ignore_patterns=ignore_patterns if ignore_patterns else None,
)

print(f"\nuploaded to https://huggingface.co/{HF_REPO_NAME}")